## MODEL

In [10]:
import torch 
from new_dataloader import (BalancedAgeBatchSampler, TestRetestDataset)
from datetime import datetime  
from torch.utils.data import DataLoader
import pandas as pd
from pathlib import Path
from tqdm import tqdm

class ResBlock(torch.nn.Module):

    def __init__(self,in_channels,out_channels):
        super().__init__()
        self.conv1=torch.nn.Conv3d(in_channels,out_channels,3,1,1)
        self.batchnorm=torch.nn.BatchNorm3d(out_channels,affine=True,track_running_stats=False) # 原 為 False
        self.ELU=torch.nn.ELU()
        self.conv2=torch.nn.Conv3d(out_channels,out_channels,3,1,1)
        self.conv_shortcut=torch.nn.Conv3d(in_channels,out_channels,1,1,0)
        self.maxpooling=torch.nn.MaxPool3d(2,2,ceil_mode=False)
    def forward(self, x):    
        x_=self.conv_shortcut(x)
        x=self.conv1(x)
        x=self.batchnorm(x)
        x=self.ELU(x)
        x=self.conv2(x)
        x=self.batchnorm(x)
        x=self.ELU(x+x_)
        return x
    
class NetResBlock(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.resblock1=ResBlock(1,8)
        self.resblock2=ResBlock(8,16)
        self.resblock3=ResBlock(16,32)
        self.resblock4=ResBlock(32,64)
        self.resblock5=ResBlock(64,128)
        self.resblock6=ResBlock(128,256)
        self.maxpooling1=torch.nn.MaxPool3d(2,2,1,ceil_mode=False)
        self.maxpooling2=torch.nn.MaxPool3d(2,2,(1,0,1),ceil_mode=False)
        
    def forward(self, x):
        x=self.resblock1(x)
        x=self.maxpooling1(x)
        x=self.resblock2(x)
        x=self.maxpooling1(x)
        x=self.resblock3(x)
        x=self.maxpooling2(x)
        x=self.resblock4(x)
        x=self.maxpooling1(x)
        x=self.resblock5(x)
        x=self.maxpooling1(x)
        x=self.resblock6(x)
        x=self.maxpooling2(x)
        return x 
    
class RESNET_MODEL(torch.nn.Module):
    def __init__(self,model_type):
        super().__init__()
        valid_types = {
            "T1",
            "T1_SEX",
            "T1_B0",
            "T1_SEX_B0",
        }
        if model_type not in valid_types:
            raise ValueError(
                f"Unknown model_type: {model_type}. "
                f"Choose from {valid_types}"
            )
        self.model_type = model_type
        self.RES=NetResBlock()
        self.flatten=torch.nn.Flatten()
        self.fc1=torch.nn.Linear(16384,256)
        self.ELU=torch.nn.ELU()
        self.dropout=torch.nn.Dropout(p=0.2)  

        self.fc2_for_no_cat = torch.nn.Linear(256, 1)  # T1-only
        cat_dim = 258 if model_type == "T1_SEX_B0" else 257   # SEX 或 B0：256 + 1 = 257 、 SEX + B0：256 + 2 = 258
        self.fc2_for_cat = torch.nn.Linear(cat_dim,1)
        
    def forward(self,x,gender,scanner):
        
        x=self.RES(x)
        x=self.flatten(x)
        x=self.fc1(x)
        x=self.ELU(x)
        x=self.dropout(x)

        if self.model_type == "T1":
            x = self.fc2_for_no_cat(x)

        elif self.model_type == "T1_SEX":
            x = torch.cat((x, gender),dim=1)
            x = self.fc2_for_cat(x)

        elif self.model_type == "T1_B0":
            x = torch.cat((x, scanner),dim=1)
            x = self.fc2_for_cat(x)

        elif self.model_type == "T1_SEX_B0":
            x = torch.cat((x, gender, scanner),dim=1)
            x = self.fc2_for_cat(x)

        return x

## PATH

In [13]:
# 2. 路徑與評估設定
ROOT = Path(r"C:\Lab\models\ResNet")
OUTPUT_DIR = ROOT / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATHS = {
    "T1": ROOT / "pretrained" / "model_20250624_095907_881.pt",
    "T1_SEX": ROOT / "pretrained" / "model_20250705_182002_540.pt",
    "T1_B0": ROOT / "pretrained" / "model_20250620_095129_948.pt",
    "T1_SEX_B0": ROOT / "pretrained" / "model_20250628_092154_757.pt",
}

DATASETS = ("test", "val")
BATCH_SIZE = 4

# 強制使用 NVIDIA CUDA GPU；若 CUDA 不可用就停止，避免意外改用 CPU。
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU 不可用，請確認 GPU 驅動與 PyTorch CUDA 版本。")

device = torch.device("cuda")
print(f"Using GPU: {torch.cuda.get_device_name(device)}")

Using GPU: NVIDIA GeForce RTX 5060 Ti


In [ ]:
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

for model_type, model_path in MODEL_PATHS.items():
    checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)
    state_dict = checkpoint.state_dict() if isinstance(checkpoint, torch.nn.Module) else checkpoint
    model = RESNET_MODEL(model_type)
    model.load_state_dict(state_dict, strict=True)
    
    # 設為評估模式
    model.to(device).eval()
    del checkpoint, state_dict

    for dataset_name in DATASETS:
        dataset = TestRetestDataset(dataset_name)
        loader = DataLoader(
            dataset, batch_size=BATCH_SIZE, shuffle=False, drop_last=False,
            num_workers=0, pin_memory=(device.type == "cuda"),
        )
        rows = []

        with torch.no_grad():
            for batch in tqdm(loader, desc=f"{dataset_name} | {model_type}"):
                images = batch["image"].to(device)
                if tuple(images.shape[1:]) != (1, 193, 229, 193):
                    raise ValueError(f"Unexpected MRI shape: {tuple(images.shape)}")

                # 保留原 test.ipynb 的輸入轉換；Scanner 1.5 仍會轉成 1。
                gender = batch["gender"].reshape(-1, 1).int().to(device)
                scanner = batch["scanner"].reshape(-1, 1).int().to(device)
                predictions = model(images, gender, scanner).reshape(-1).cpu()
                if not torch.isfinite(predictions).all():
                    raise ValueError(f"Non-finite predictions: {dataset_name} / {model_type}")

                rows.extend(zip(
                    batch["subject_name"], batch["age"].tolist(), predictions.tolist()
                ))

        result_df = pd.DataFrame(rows, columns=["Subject ID", "True Age", "Predicted Age"])
        if len(result_df) != len(dataset):
            raise RuntimeError("Prediction count mismatch")

        filename = f"ResNet_{dataset_name}_{model_type}_sequential_BS{BATCH_SIZE}_{timestamp}.xlsx"
        output_path = OUTPUT_DIR / filename
        result_df.to_excel(output_path, index=False)

        mae = (result_df["Predicted Age"] - result_df["True Age"]).abs().mean()
        print(f"N={len(result_df)} | MAE={mae:.3f} | Saved: {output_path}")

    del model


test | T1: 100%|██████████| 88/88 [00:32<00:00,  2.72it/s]


N=350 | MAE=8.676 | Saved: C:\Lab\models\ResNet\output\ResNet_test_T1_sequential_BS4_20261029_151322.xlsx


val | T1: 100%|██████████| 88/88 [00:34<00:00,  2.57it/s]


N=350 | MAE=8.107 | Saved: C:\Lab\models\ResNet\output\ResNet_val_T1_sequential_BS4_20261029_151322.xlsx


test | T1_SEX: 100%|██████████| 88/88 [00:29<00:00,  3.00it/s]


N=350 | MAE=9.183 | Saved: C:\Lab\models\ResNet\output\ResNet_test_T1_SEX_sequential_BS4_20261029_151322.xlsx


... (6 outputs left out)

test | T1_SEX_B0: 100%|██████████| 88/88 [00:28<00:00,  3.06it/s]


N=350 | MAE=9.428 | Saved: C:\Lab\models\ResNet\output\ResNet_test_T1_SEX_B0_sequential_BS4_20261029_151322.xlsx


val | T1_SEX_B0: 100%|██████████| 88/88 [00:27<00:00,  3.15it/s]

N=350 | MAE=9.245 | Saved: C:\Lab\models\ResNet\output\ResNet_val_T1_SEX_B0_sequential_BS4_20261029_151322.xlsx


In [16]:
from pathlib import Path
import re
import pandas as pd

output_dir = Path(r"C:\Lab\models\ResNet\output")

# 每個預測 Excel 對應一個工作表；順序也會作為輸出工作表的排列順序。
sheet_names = [
    "ResNet_test_T1",
    "ResNet_test_T1_SEX",
    "ResNet_test_T1_B0",
    "ResNet_test_T1_SEX_B0",
    "ResNet_val_T1",
    "ResNet_val_T1_SEX",
    "ResNet_val_T1_B0",
    "ResNet_val_T1_SEX_B0",
]

# 從檔名解析工作表名稱、批次大小與執行時間。
filename_pattern = re.compile(
    r"^(?P<sheet>ResNet_(?:test|val)_(?:T1_SEX_B0|T1_SEX|T1_B0|T1))"
    r"_sequential_BS(?P<batch_size>\d+)_(?P<timestamp>\d{8}_\d{6})$"
)
files_by_run = {}
for file_path in output_dir.glob("ResNet_*_sequential_BS*_*.xlsx"):
    match = filename_pattern.fullmatch(file_path.stem)
    if match:
        run_key = (match.group("timestamp"), int(match.group("batch_size")))
        sheet_name = match.group("sheet")
        files_by_run.setdefault(run_key, {})[sheet_name] = file_path

# 選最新且包含所有 8 種結果的同一批檔案，避免混合不同次執行或批次大小。
complete_runs = [
    run_key
    for run_key, files in files_by_run.items()
    if all(sheet_name in files for sheet_name in sheet_names)
]
if not complete_runs:
    raise FileNotFoundError(
        f"找不到同一時間戳且包含全部 {len(sheet_names)} 個預測檔的完整資料組。"
    )

latest_run = max(complete_runs, key=lambda run: (run[0], run[1]))
latest_timestamp, batch_size = latest_run
files = files_by_run[latest_run]
save_path = output_dir / f"ResNet_NoAgeBatch_BN{batch_size}_{latest_timestamp}.xlsx"

# 將選定批次的 8 個 Excel 依指定名稱合併為單一活頁簿。
with pd.ExcelWriter(save_path, engine="openpyxl") as writer:
    for sheet_name in sheet_names:
        df = pd.read_excel(files[sheet_name])
        df.to_excel(writer, sheet_name=sheet_name, index=False)

print(f"Merged timestamp: {latest_timestamp} | Batch size: {batch_size}")
print(f"Saved: {save_path}")

Merged timestamp: 20261029_151322 | Batch size: 4
Saved: C:\Lab\models\ResNet\output\ResNet_NoAgeBatch_BN4_20261029_151322.xlsx
